# Model Building and Evaluation

INT 374 - PCOS Prediction Project

Training multiple machine learning models and comparing their performance.
Using metrics like accuracy, precision, recall, F1-score, and AUC.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.metrics import confusion_matrix, roc_curve, auc, precision_recall_curve
import warnings

warnings.filterwarnings('ignore')
print('Libraries loaded')

Libraries loaded


In [3]:
# Load data
df = pd.read_csv('../data/PCOS_final.csv')
print(f'Loaded: {df.shape}')

X = df.drop('PCOS', axis=1)
y = df['PCOS']
print(f'Features: {X.shape[1]}')
print(f'Target distribution: {y.value_counts().to_dict()}')

Loaded: (541, 33)
Features: 32
Target distribution: {0: 364, 1: 177}


## Train-Test Split and Scaling

In [4]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train set: {X_train.shape}')
print(f'Test set: {X_test.shape}')
print(f'Train PCOS rate: {y_train.mean()*100:.1f}%')
print(f'Test PCOS rate: {y_test.mean()*100:.1f}%')

Train set: (432, 32)
Test set: (109, 32)
Train PCOS rate: 32.6%
Test PCOS rate: 33.0%


In [5]:
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('Data scaled')

Data scaled


## Train Models

In [ ]:
# Define models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=10, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42),
    'SVM': SVC(kernel='rbf', probability=True, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'XGBoost': XGBClassifier(n_estimators=100, max_depth=5, random_state=42, verbosity=0)
}

print(f'Training {len(models)} models...')

In [ ]:
# Train and evaluate models
results = {}

for name, model in models.items():
    # Use scaled data for SVM and KNN, original for tree-based
    if name in ['SVM', 'Logistic Regression', 'KNN']:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    # Calculate metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc_score = roc_auc_score(y_test, y_pred_proba)
    
    results[name] = {
        'model': model,
        'y_pred': y_pred,
        'y_pred_proba': y_pred_proba,
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1': f1,
        'auc': auc_score
    }
    
    print(f'{name}: Acc={acc:.3f}, Prec={prec:.3f}, Rec={rec:.3f}, F1={f1:.3f}, AUC={auc_score:.3f}')

## Model Comparison

In [ ]:
# Create comparison dataframe
comp_df = pd.DataFrame({
    'Model': results.keys(),
    'Accuracy': [results[m]['accuracy'] for m in results],
    'Precision': [results[m]['precision'] for m in results],
    'Recall': [results[m]['recall'] for m in results],
    'F1-Score': [results[m]['f1'] for m in results],
    'AUC': [results[m]['auc'] for m in results]
})

print('\nModel Comparison:')
print(comp_df.to_string(index=False))

best_model = comp_df.loc[comp_df['F1-Score'].idxmax(), 'Model']
print(f'\nBest model (by F1-Score): {best_model}')

In [ ]:
# Plot model comparison
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC']
x = np.arange(len(metrics))
width = 0.13

fig, ax = plt.subplots(figsize=(12, 6))

for i, model in enumerate(results.keys()):
    values = [
        results[model]['accuracy'],
        results[model]['precision'],
        results[model]['recall'],
        results[model]['f1'],
        results[model]['auc']
    ]
    ax.bar(x + i*width, values, width, label=model)

ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison')
ax.set_xticks(x + width * 2.5)
ax.set_xticklabels(metrics)
ax.legend(fontsize=8)
ax.set_ylim(0.6, 1.0)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../outputs/figures/31_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved model comparison plot')

## Confusion Matrices

## Confusion Matrices

In [ ]:
# Plot confusion matrices
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for idx, (name, result) in enumerate(results.items()):
    cm = confusion_matrix(y_test, result['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx], 
                xticklabels=['No PCOS', 'PCOS'],
                yticklabels=['No PCOS', 'PCOS'])
    acc = results[name]['accuracy']
    f1 = results[name]['f1']
    axes[idx].set_title(f'{name}\nAcc={acc:.2f}, F1={f1:.2f}')
    axes[idx].set_ylabel('True')
    axes[idx].set_xlabel('Predicted')

plt.tight_layout()
plt.savefig('../outputs/figures/28_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved confusion matrices')

## ROC Curves

In [ ]:
# Plot ROC curves
fig, ax = plt.subplots(figsize=(10, 8))

colors = ['red', 'orange', 'green', 'cyan', 'blue', 'magenta']

for idx, (name, result) in enumerate(results.items()):
    fpr, tpr, _ = roc_curve(y_test, result['y_pred_proba'])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=colors[idx], lw=2, 
            label=f'{name} (AUC={roc_auc:.3f})')

# Diagonal line
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random')

ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves - Model Comparison')
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/figures/29_roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved ROC curves')

## Precision-Recall Curves

In [ ]:
# Plot precision-recall curves
fig, ax = plt.subplots(figsize=(10, 8))

for idx, (name, result) in enumerate(results.items()):
    precision, recall, _ = precision_recall_curve(y_test, result['y_pred_proba'])
    ax.plot(recall, precision, color=colors[idx], lw=2, label=name)

ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curves')
ax.legend(loc='best', fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig('../outputs/figures/30_precision_recall_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved precision-recall curves')

## Save Best Model

In [ ]:
# Save best model (Random Forest has highest F1 and AUC)
import pickle

best_model_obj = results['Random Forest']['model']
pickle.dump(best_model_obj, open('../outputs/models/best_model.pkl', 'wb'))
pickle.dump(scaler, open('../outputs/models/scaler.pkl', 'wb'))
pickle.dump(X.columns.tolist(), open('../outputs/models/feature_names.pkl', 'wb'))

print('Saved models and scaler')

In [ ]:
# Save results
comp_df.to_csv('../outputs/reports/model_comparison_report.csv', index=False)
print('Saved: model_comparison_report.csv')
print('\nModel building completed')